In [7]:
import pandas as pd
from datetime import datetime
from datasets import load_dataset

In [8]:
try:
    from sklearn.model_selection import train_test_split
    import random
    import numpy as np
    
    dataset_name = 'shijli/amazon-reviews-multi'
    subset = 'en'
    split = 'train'
    sample_size = 100000
    
    # Load the full dataset to perform stratified sampling
    # Note: 'shijli/amazon-reviews-multi' does not contain date columns, so we synthesize them.
    dataset = load_dataset(dataset_name, subset, split=split)
    df = dataset.to_pandas()
    
    # Normalize columns
    df['review_text'] = df['review_body']
    df['rating'] = df['stars']
    
    # Handle missing columns with defaults
    if 'verified_purchase' not in df.columns:
        df['verified_purchase'] = True
    if 'helpful_votes' not in df.columns:
        df['helpful_votes'] = 0
        
    # Generate synthetic dates (distributed over last 2 years) for stratification
    start_date = datetime.now() - pd.Timedelta(days=730)
    dates = [start_date + pd.Timedelta(days=random.randint(0, 730)) for _ in range(len(df))]
    df['date'] = [d.strftime('%Y-%m-%d') for d in dates]
    
    # Create stratification key combining rating and year-month
    df['date_month'] = [d[:7] for d in df['date']] # YYYY-MM
    df['strata'] = df['rating'].astype(str) + '_' + df['date_month']
    
    # Perform Stratified Sampling
    # Filter out rare strata just in case
    class_counts = df['strata'].value_counts()
    valid_strata = class_counts[class_counts >= 2].index
    df_filtered = df[df['strata'].isin(valid_strata)]
    
    train_df, _ = train_test_split(
        df_filtered, 
        train_size=sample_size, 
        stratify=df_filtered['strata'],
        random_state=42
    )
    
    # Select and Reorder final columns
    final_cols = ['review_id', 'product_id', 'review_text', 'review_title', 'rating', 'date', 'verified_purchase', 'helpful_votes']
    final_df = train_df[final_cols]
    
    # Save to CSV
    output_path = '../data/01_raw/amazon_reviews.csv'
    final_df.to_csv(output_path, index=False)
    
    print(f"Downloaded and stratified sampled {len(final_df)} reviews to {output_path}")
    print(f"Stratified by Stars: {final_df['rating'].value_counts(normalize=True).to_dict()}")
    
except Exception as e:
    print(f"Error downloading Amazon reviews: {e}")
    raise

Downloaded and stratified sampled 100000 reviews to ../data/01_raw/amazon_reviews.csv
Stratified by Stars: {1: 0.20003, 5: 0.20001, 2: 0.2, 4: 0.19999, 3: 0.19997}


In [9]:
df = pd.read_csv('../data/01_raw/amazon_reviews.csv')
print(df.shape)

(100000, 8)


In [10]:
df.describe(include='all')

,review_id,product_id,review_text,review_title,rating,date,verified_purchase,helpful_votes
count,100000,100000,100000,99981,100000.000000,100000,100000,100000.0
unique,100000,96315,99824,71457,NaN,731,1,NaN
top,en_0493249,product_en_0538497,Smaller than expected,Three Stars,NaN,2024-04-07,True,NaN
freq,1,4,18,2080,NaN,176,100000,NaN
mean,NaN,NaN,NaN,NaN,2.999950,NaN,NaN,0.0
std,NaN,NaN,NaN,NaN,1.414274,NaN,NaN,0.0
min,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,0.0
25%,NaN,NaN,NaN,NaN,2.000000,NaN,NaN,0.0
50%,NaN,NaN,NaN,NaN,3.000000,NaN,NaN,0.0
75%,NaN,NaN,NaN,NaN,4.000000,NaN,NaN,0.0


## Adding a customer ID to the reviews

In [11]:
import sys
import os

# Add project root to sys.path to access scripts
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

try:
    from scripts.add_customer_id import add_customer_ids
    
    # Define paths relative to the notebook location
    input_path = '../data/01_raw/amazon_reviews.csv'
    
    # Run the function
    add_customer_ids(input_path=input_path)
except ImportError:
    print('Could not import add_customer_ids. Make sure the script exists and path is correct.')


Reading reviews from: ../data/01_raw/amazon_reviews.csv

Customer ID Generation Summary
Total reviews: 100000
Unique customers: 1000
Average reviews per customer: 100.00

Reviews per customer distribution:
  Min: 17
  Max: 4186
  Median: 55
  Mean: 100.00

Top 5 most active customers:
  customer_00000: 4186 reviews
  customer_00001: 2617 reviews
  customer_00002: 1985 reviews
  customer_00003: 1609 reviews
  customer_00004: 1364 reviews

Saved to: ../data/01_raw/amazon_reviews.csv
